# A tour of PARIS

**PARIS** — Portfolio Analytics, Risk & Investment Statistics — is a small, pure-Python
(numpy + pandas) library of performance and risk statistics for fund return series, absolute and
benchmark-relative, plus portfolio construction from weights, contribution and Brinson attribution.

This notebook walks through the **most used** statistics, one cell each, in the most general call
form: a DataFrame of funds in, a Series indexed by fund out. It does not show every function or every
keyword switch — every function's docstring documents its formula and switches (`help(paris.sharpe)`),
and every number is cross-checked against independent reference implementations and public
portfolio tools. The sections below follow the library's topic modules.

Everything runs on the sample data that ships with the package (`paris.data`).

*PARIS is for internal analytical, educational and research use only; see `DISCLAIMER.md`.*

## 1. Inputs, alignment and conventions

Inputs are periodic **simple** returns as pandas Series/DataFrames with a `DatetimeIndex`. The setup
below is the one every example uses: six active US large-cap funds, the S&P 500
ETF as benchmark and the 3-month T-bill as a per-period risk-free series, all monthly 2010–2025; plus a
daily SPY series 2021–2025.

In [1]:
import numpy as np
import pandas as pd

import paris

np.set_printoptions(legacy="1.25")  # display floats as plain numbers rather than np.float64(...)

m = paris.data.load_managers()                 # monthly total returns, 2010-01 .. 2025-12
rets  = m[["FCNTX", "AGTHX", "FMAGX", "AMCPX", "DODGX", "PRGFX"]]   # DataFrame: the six funds
fund  = m["FCNTX"]                             # Series: one fund
spx   = m["SPY"]                               # Series: benchmark
tbill = m["TBILL3M"]                           # Series: risk-free, already per month

px    = paris.data.load_prices()               # daily total-return index levels, 2021 .. 2025
daily = px["SPY"].pct_change().dropna()        # Series: daily simple returns

print(paris.__version__)
rets.tail(3)

0.4.2


,FCNTX,AGTHX,FMAGX,AMCPX,DODGX,PRGFX
date,,,,,,
2025-10-31,0.010097,0.024937,-0.002484,0.028602,-0.006318,0.035079
2025-11-30,-0.000800,-0.008888,-0.016189,0.006225,0.012717,-0.015826
2025-12-31,0.016607,0.001037,-0.010611,-0.005858,0.018433,-0.009463


**Series in → scalar out; DataFrame in → Series out.** Every statistic has the same shape rule, so a
whole fund universe is one call.

In [2]:
paris.sharpe(fund), paris.sharpe(rets)

(1.0434655230078411,
 FCNTX    1.043466
 AGTHX    0.918000
 FMAGX    0.831776
 AMCPX    0.881168
 DODGX    0.800608
 PRGFX    0.882514
 dtype: float64)

**Risk-free rate (§1.4).** A scalar `rf` is an **annual** rate and is de-annualised geometrically to the
observation frequency; a Series `rf` is taken as already **per period** and aligned by date. The
difference matters: the T-bill averaged well under 2 % over 2010–2025.

In [3]:
pd.DataFrame({"rf = 2 % p.a.": paris.sharpe(rets, rf=0.02),
              "rf = T-bill series": paris.sharpe(rets, rf=tbill)})

,rf = 2 % p.a.,rf = T-bill series
FCNTX,0.911219,0.951415
AGTHX,0.791565,0.829101
FMAGX,0.707477,0.744000
AMCPX,0.747699,0.786785
DODGX,0.680089,0.714116
PRGFX,0.766220,0.800508


**Frequency (§1.3).** The number of periods per year, $m$, is inferred from the index spacing — 12 for
the monthly frame, 252 for the daily series — and drives every annualisation. `periods_per_year`
overrides it.

In [4]:
print("monthly funds, m = 12 inferred :", paris.volatility(rets).round(4).to_dict())
print("daily SPY,     m = 252 inferred:", round(paris.volatility(daily), 4))
print("daily SPY,     m = 260 forced  :", round(paris.volatility(daily, periods_per_year=260), 4))

monthly funds, m = 12 inferred : {'FCNTX': 0.1499, 'AGTHX': 0.1568, 'FMAGX': 0.1594, 'AMCPX': 0.1485, 'DODGX': 0.1644, 'PRGFX': 0.1704}
daily SPY,     m = 252 inferred: 0.1711
daily SPY,     m = 260 forced  : 0.1738


**Gap policy (§1.2).** Inputs are trimmed to their common window; inside it, any NaN — or any
benchmark / rf date absent from the returns index — raises `GapError` (a `ValueError` subclass, like every
PARIS error). Nothing is ever filled, interpolated or dropped: resolve gaps upstream, then call PARIS.

In [5]:
with_gap = rets.copy()
with_gap.loc["2018-05-31", "FCNTX"] = np.nan        # one interior observation missing
try:
    paris.sharpe(with_gap)
except paris.GapError as e:
    print("GapError:", e)

GapError: returns has 1 interior gap(s) in ['FCNTX'] (e.g. 2018-05-31); gaps must be resolved upstream, PARIS never fills or drops them


## 2. Returns

Compounding is geometric by default (`geometric=False` switches to arithmetic sums where it applies).

**`wealth_index`** — growth of 1 unit invested at the start of the sample (§2).

In [6]:
paris.wealth_index(rets).tail()

,FCNTX,AGTHX,FMAGX,AMCPX,DODGX,PRGFX
date,,,,,,
2025-08-31,9.565734,7.762731,6.869991,6.399164,6.385819,8.239140
2025-09-30,9.815481,7.993938,6.952034,6.550160,6.399736,8.614003
2025-10-31,9.914588,8.193285,6.934762,6.737507,6.359302,8.916174
2025-11-30,9.906659,8.120464,6.822493,6.779451,6.440171,8.775066
2025-12-31,10.071178,8.128883,6.750101,6.739738,6.558880,8.692028


**`total_return`** — compounded return over the whole sample.

In [7]:
paris.total_return(rets)

FCNTX    9.071178
AGTHX    7.128883
FMAGX    5.750101
AMCPX    5.739738
DODGX    5.558880
PRGFX    7.692028
dtype: float64

**`annualized_return`** — geometric annualised return, $(1+R)^{m/n}-1$; this is the `CAGR` row of the
summary table. (`cagr` is the calendar-day variant used by the public web tools.)

In [8]:
paris.annualized_return(rets)

FCNTX    0.155294
AGTHX    0.139927
FMAGX    0.126761
AMCPX    0.126653
DODGX    0.124739
PRGFX    0.144709
dtype: float64

**`calendar_returns`** — returns compounded into calendar periods (years by default; `freq="QE"` for quarters).

In [9]:
paris.calendar_returns(rets).tail()

,FCNTX,AGTHX,FMAGX,AMCPX,DODGX,PRGFX
date,,,,,,
2021,0.244363,0.193365,0.269831,0.236570,0.316825,0.200315
2022,-0.282613,-0.307199,-0.271513,-0.287801,-0.072452,-0.401439
2023,0.393324,0.372011,0.309472,0.309758,0.174852,0.452743
2024,0.359712,0.284254,0.280322,0.211309,0.144941,0.295939
2025,0.217542,0.199335,0.105769,0.177745,0.136808,0.156503


**`best` / `worst`** — the extreme single-period returns.

In [10]:
pd.DataFrame({"best": paris.best(rets), "worst": paris.worst(rets)})

,best,worst
FCNTX,0.145408,-0.115569
AGTHX,0.142824,-0.120121
FMAGX,0.129398,-0.116139
AMCPX,0.134678,-0.117572
DODGX,0.183154,-0.196721
PRGFX,0.143291,-0.152995


**`win_rate`** — share of periods with a positive return (zeros excluded by default).

In [11]:
paris.win_rate(rets)

FCNTX    0.656250
AGTHX    0.645833
FMAGX    0.640625
AMCPX    0.666667
DODGX    0.625000
PRGFX    0.625000
dtype: float64

**`period_returns`** — trailing and to-date returns as of the last observation: month-, quarter- and
year-to-date, then 1/3/5/10-year and inception-to-date (annualised beyond one year).

In [12]:
paris.period_returns(rets)

,FCNTX,AGTHX,FMAGX,AMCPX,DODGX,PRGFX
MTD,0.016607,0.001037,-0.010611,-0.005858,0.018433,-0.009463
QTD,0.026050,0.016881,-0.029047,0.028942,0.024867,0.009058
YTD,0.217542,0.199335,0.105769,0.177745,0.136808,0.156503
1Y,0.217542,0.199335,0.105769,0.177745,0.136808,0.156503
3Y,0.321278,0.283265,0.228457,0.231683,0.152085,0.296105
5Y,0.155411,0.118063,0.113909,0.104749,0.133088,0.093616
10Y,0.168352,0.151498,0.137584,0.126622,0.127966,0.141569
ITD,0.155294,0.139927,0.126761,0.126653,0.124739,0.144709


**`aggregate`** — compound a series to a lower frequency, e.g. daily → month-end.

In [13]:
paris.aggregate(daily, "ME").tail()

date
2025-08-31    0.020520
2025-09-30    0.035606
2025-10-31    0.023837
2025-11-30    0.001950
2025-12-31    0.000772
Freq: ME, Name: SPY, dtype: float64

## 3. Risk: dispersion, moments and tails

Standard deviations use the sample divisor (`ddof=1`); VaR and CVaR are returned as (negative) returns
at the 95 % level by default and are per period.

**`volatility`** — annualised standard deviation, $s\sqrt{m}$.

In [14]:
paris.volatility(rets)

FCNTX    0.149863
AGTHX    0.156751
FMAGX    0.159446
AMCPX    0.148492
DODGX    0.164447
PRGFX    0.170421
dtype: float64

**`downside_deviation`** — root-mean-square of the shortfall below a per-period target `mar`
(default 0), over **all** observations (`method="full"`); per period unless `annualize=True`.

In [15]:
paris.downside_deviation(rets)

FCNTX    0.025593
AGTHX    0.027563
FMAGX    0.028915
AMCPX    0.026895
DODGX    0.029879
PRGFX    0.030432
dtype: float64

**`skewness`** — third standardised moment (population form, `method="moment"`).

In [16]:
paris.skewness(rets)

FCNTX   -0.299305
AGTHX   -0.285086
FMAGX   -0.360750
AMCPX   -0.390879
DODGX   -0.365005
PRGFX   -0.352969
dtype: float64

**`kurtosis`** — **excess** kurtosis (population form); 0 for a normal distribution.

In [17]:
paris.kurtosis(rets)

FCNTX    0.384693
AGTHX    0.529986
FMAGX    0.286320
AMCPX    0.651507
DODGX    2.126106
PRGFX    0.632034
dtype: float64

**`var`** — value at risk as the empirical 5 % return quantile (`method="historical"`). The other
common convention, the Cornish–Fisher expansion, is `method="modified"`.

In [18]:
pd.DataFrame({"historical": paris.var(rets),
              "modified": paris.var(rets, method="modified")})

,historical,modified
FCNTX,-0.063048,-0.061206
AGTHX,-0.075210,-0.065350
FMAGX,-0.070368,-0.068790
AMCPX,-0.068567,-0.063486
DODGX,-0.071126,-0.069671
PRGFX,-0.084330,-0.072359


**`cvar`** — conditional VaR / expected shortfall: the mean of the returns at or below the VaR quantile.

In [19]:
paris.cvar(rets)

FCNTX   -0.085832
AGTHX   -0.093040
FMAGX   -0.094856
AMCPX   -0.090274
DODGX   -0.098728
PRGFX   -0.101233
dtype: float64

**`tail_ratio`** — 95th percentile return divided by the absolute 5th percentile; > 1 means the right tail is fatter.

In [20]:
paris.tail_ratio(rets)

FCNTX    1.305788
AGTHX    1.130538
FMAGX    1.158137
AMCPX    1.039154
DODGX    1.076990
PRGFX    1.084162
dtype: float64

## 4. Drawdowns

Drawdowns are measured from the running peak of the geometric wealth index. The drawdown series and
the maximum drawdown are **negative** numbers; averaged depths (`avg_drawdown`, `ulcer_index`, `pain_index`)
are reported as positive magnitudes, as in the R reference package. Lengths and recoveries are counted in
observations.

**`drawdowns`** — the drawdown series itself, one column per fund.

In [21]:
paris.drawdowns(rets).tail()

,FCNTX,AGTHX,FMAGX,AMCPX,DODGX,PRGFX
date,,,,,,
2025-08-31,0.0000,0.000000,-0.008105,0.000000,0.000000,0.000000
2025-09-30,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000
2025-10-31,0.0000,0.000000,-0.002484,0.000000,-0.006318,0.000000
2025-11-30,-0.0008,-0.008888,-0.018634,0.000000,0.000000,-0.015826
2025-12-31,0.0000,-0.007860,-0.029047,-0.005858,0.000000,-0.025139


**`max_drawdown`** — the deepest peak-to-trough loss in the sample.

In [22]:
paris.max_drawdown(rets)

FCNTX   -0.309458
AGTHX   -0.335673
FMAGX   -0.307234
AMCPX   -0.324500
DODGX   -0.291576
PRGFX   -0.407253
dtype: float64

**`drawdown_table`** — the worst episodes of one fund: peak, trough, recovery dates, depth and lengths.

In [23]:
paris.drawdown_table(fund, top=5)

,start,trough,end,depth,length,to_trough,recovery
0,2022-01-31,2022-09-30,2024-01-31,-0.309458,25,9,16
1,2018-10-31,2018-12-31,2019-04-30,-0.162369,7,3,4
2,2020-02-29,2020-03-31,2020-05-31,-0.154342,4,2,2
3,2011-05-31,2011-09-30,2012-02-29,-0.144033,10,5,5
4,2010-05-31,2010-06-30,2010-09-30,-0.096248,5,2,3


**`avg_drawdown` / `longest_drawdown`** — mean depth over all episodes (a positive magnitude), and the longest episode in periods.

In [24]:
pd.DataFrame({"avg_drawdown": paris.avg_drawdown(rets), "longest (periods)": paris.longest_drawdown(rets)})

,avg_drawdown,longest (periods)
FCNTX,0.052589,25.0
AGTHX,0.053131,28.0
FMAGX,0.056329,26.0
AMCPX,0.057422,26.0
DODGX,0.054767,18.0
PRGFX,0.063789,32.0


**`ulcer_index`** — root-mean-square drawdown over the sample.

In [25]:
paris.ulcer_index(rets)

FCNTX    0.075679
AGTHX    0.090013
FMAGX    0.085770
AMCPX    0.084247
DODGX    0.061398
PRGFX    0.108503
dtype: float64

**`calmar_ratio`** — annualised return divided by the absolute maximum drawdown, over the **full** sample.

In [26]:
paris.calmar_ratio(rets)

FCNTX    0.501825
AGTHX    0.416854
FMAGX    0.412588
AMCPX    0.390302
DODGX    0.427811
PRGFX    0.355329
dtype: float64

## 5. Risk-adjusted ratios

Ratios are annualised where the literature annualises them; `rf` follows §1.4 and `mar` is per period.

**`sharpe`** — annualised mean excess return over annualised excess-return volatility (arithmetic
numerator, the current default of the R reference package and of the public tools).

In [27]:
paris.sharpe(rets, rf=tbill)

FCNTX    0.951415
AGTHX    0.829101
FMAGX    0.744000
AMCPX    0.786785
DODGX    0.714116
PRGFX    0.800508
dtype: float64

**`sortino`** — annualised mean return above `mar` (default 0) over the annualised downside deviation.

In [28]:
paris.sortino(rets)

FCNTX    1.763846
AGTHX    1.507104
FMAGX    1.324052
AMCPX    1.404447
DODGX    1.271998
PRGFX    1.426661
dtype: float64

**`omega`** — probability-weighted gains above `mar` divided by losses below it.

In [29]:
paris.omega(rets)

FCNTX    2.140628
AGTHX    1.966890
FMAGX    1.827681
AMCPX    1.911620
DODGX    1.831275
PRGFX    1.934481
dtype: float64

**`profit_factor`** — sum of gains divided by the absolute sum of losses.

In [30]:
paris.profit_factor(rets)

FCNTX    2.140628
AGTHX    1.966890
FMAGX    1.827681
AMCPX    1.911620
DODGX    1.831275
PRGFX    1.934481
dtype: float64

**`payoff_ratio`** — mean gain divided by the absolute mean loss.

In [31]:
paris.payoff_ratio(rets)

FCNTX    1.121281
AGTHX    1.078617
FMAGX    1.025285
AMCPX    0.955810
DODGX    1.098765
PRGFX    1.160688
dtype: float64

**`kelly_ratio`** — continuous Kelly leverage, mean excess return over variance; `half=True` is the commonly quoted half-Kelly.

In [32]:
paris.kelly_ratio(rets, rf=tbill, half=True)

FCNTX    3.167132
AGTHX    2.640962
FMAGX    2.330707
AMCPX    2.646968
DODGX    2.173255
PRGFX    2.346203
dtype: float64

**`probabilistic_sharpe`** — probability that the true Sharpe ratio exceeds `benchmark_sharpe` (0 by default), given skewness, kurtosis and sample size.

In [33]:
paris.probabilistic_sharpe(rets, rf=tbill)

FCNTX    0.999821
AGTHX    0.999166
FMAGX    0.997625
AMCPX    0.998456
DODGX    0.996477
PRGFX    0.998737
dtype: float64

## 6. Benchmark-relative statistics

The benchmark is the second positional argument. Regression statistics are estimated on **excess**
returns (both sides net of `rf`); capture and tracking statistics on raw returns.

**`beta`** — slope of the CAPM regression of fund excess returns on benchmark excess returns.

In [34]:
paris.beta(rets, spx, rf=tbill)

FCNTX    0.982919
AGTHX    1.046639
FMAGX    1.063388
AMCPX    1.001575
DODGX    1.052794
PRGFX    1.095672
dtype: float64

**`alpha`** — the regression intercept, annualised (geometrically by default).

In [35]:
paris.alpha(rets, spx, rf=tbill)

FCNTX    0.016515
AGTHX   -0.004240
FMAGX   -0.017524
AMCPX   -0.011468
DODGX   -0.017140
PRGFX   -0.004018
dtype: float64

**`regression_stats`** — the full CAPM regression table for one fund (add the `scipy` extra for p-values).

In [36]:
paris.regression_stats(fund, spx, rf=tbill)

,alpha,beta,r2,alpha_t,beta_t,alpha_p,beta_p,resid_sd,n
FCNTX,0.001366,0.982919,0.88709,1.260052,38.636128,0.209196,6.272502e-92,0.014542,192.0


**`correlation` / `r_squared`** — correlation of raw returns and the $R^2$ of the excess-return regression.

In [37]:
pd.DataFrame({"correlation": paris.correlation(rets, spx), "r_squared": paris.r_squared(rets, spx, rf=tbill)})

,correlation,r_squared
FCNTX,0.942094,0.887090
AGTHX,0.958183,0.917800
FMAGX,0.956661,0.914960
AMCPX,0.967309,0.935574
DODGX,0.916204,0.839865
PRGFX,0.922349,0.850306


**`tracking_error`** — annualised standard deviation of the active return $r - b$.

In [38]:
paris.tracking_error(rets, spx)

FCNTX    0.050304
AGTHX    0.045374
FMAGX    0.047330
AMCPX    0.037659
DODGX    0.066300
PRGFX    0.067280
dtype: float64

**`information_ratio`** — annualised active return over tracking error.

In [39]:
paris.information_ratio(rets, spx)

FCNTX    0.299674
AGTHX   -0.006449
FMAGX   -0.284349
AMCPX   -0.360243
DODGX   -0.233486
PRGFX    0.066730
dtype: float64

**`treynor_ratio`** — annualised excess return per unit of beta.

In [40]:
paris.treynor_ratio(rets, spx, rf=tbill)

FCNTX    0.141744
AGTHX    0.118600
FMAGX    0.104501
AMCPX    0.110838
DODGX    0.103609
PRGFX    0.117597
dtype: float64

**`m_squared`** — Modigliani–Modigliani: the fund's return re-levered to the benchmark's volatility.

In [41]:
paris.m_squared(rets, spx, rf=tbill)

FCNTX    0.149170
AGTHX    0.129184
FMAGX    0.115404
AMCPX    0.122766
DODGX    0.110563
PRGFX    0.123983
dtype: float64

**`up_capture` / `down_capture`** — compounded fund return over compounded benchmark return in the
periods when the benchmark rose (fell). Above 1 up and below 1 down is the desirable pattern.

In [42]:
pd.DataFrame({"up_capture": paris.up_capture(rets, spx), "down_capture": paris.down_capture(rets, spx)})

,up_capture,down_capture
FCNTX,1.079566,0.982469
AGTHX,1.117339,1.013076
FMAGX,1.030588,1.024007
AMCPX,0.880008,1.007716
DODGX,1.035983,1.027263
PRGFX,1.252908,1.017986


**`batting_average`** — share of periods in which the fund beat the benchmark.

In [43]:
paris.batting_average(rets, spx)

FCNTX    0.531250
AGTHX    0.510417
FMAGX    0.473958
AMCPX    0.484375
DODGX    0.526042
PRGFX    0.531250
dtype: float64

**`bull_beta` / `bear_beta`** — beta estimated on the periods when the benchmark excess return was positive (negative).

In [44]:
pd.DataFrame({"bull_beta": paris.bull_beta(rets, spx, rf=tbill), "bear_beta": paris.bear_beta(rets, spx, rf=tbill)})

,bull_beta,bear_beta
FCNTX,0.985270,1.019989
AGTHX,1.070088,1.047923
FMAGX,1.071386,1.062303
AMCPX,0.993603,1.027528
DODGX,1.032370,1.114793
PRGFX,1.115683,1.190425


## 7. Summary table and Portfolio wrapper

`stats` assembles one table from the functions of chapters 2–6 with their defaults — it computes
nothing of its own, so every cell is reproducible by the direct call listed in the manual. The benchmark
gets its own last column.

In [45]:
paris.stats(rets, benchmark=spx, rf=tbill)

,FCNTX,AGTHX,FMAGX,AMCPX,DODGX,PRGFX,SPY
metric,,,,,,,
Start,2010-01-31 00:00:00,2010-01-31 00:00:00,2010-01-31 00:00:00,2010-01-31 00:00:00,2010-01-31 00:00:00,2010-01-31 00:00:00,2010-01-31 00:00:00
End,2025-12-31 00:00:00,2025-12-31 00:00:00,2025-12-31 00:00:00,2025-12-31 00:00:00,2025-12-31 00:00:00,2025-12-31 00:00:00,2025-12-31 00:00:00
Periods,192,192,192,192,192,192,192
Total Return,9.071178,7.128883,5.750101,5.739738,5.55888,7.692028,7.162334
CAGR,0.155294,0.139927,0.126761,0.126653,0.124739,0.144709,0.140219
Volatility (ann.),0.149863,0.156751,0.159446,0.148492,0.164447,0.170421,0.143359
Semi Deviation,0.031917,0.03334,0.034311,0.032057,0.035083,0.036427,0.030924
Gain Deviation,0.026732,0.027927,0.02702,0.025771,0.029373,0.030101,0.025116
Loss Deviation,0.028516,0.030537,0.030981,0.02994,0.03403,0.034746,0.027667


A subset of rows, by label (the labels are the keys of `paris.ABSOLUTE_METRICS` / `paris.RELATIVE_METRICS`).

In [46]:
paris.stats(rets, benchmark=spx, rf=tbill, metrics=["CAGR", "Volatility (ann.)", "Sharpe", "Max Drawdown", "Beta", "Information Ratio"])

,FCNTX,AGTHX,FMAGX,AMCPX,DODGX,PRGFX,SPY
metric,,,,,,,
CAGR,0.155294,0.139927,0.126761,0.126653,0.124739,0.144709,0.140219
Volatility (ann.),0.149863,0.156751,0.159446,0.148492,0.164447,0.170421,0.143359
Sharpe,0.951415,0.829101,0.744000,0.786785,0.714116,0.800508,0.893760
Max Drawdown,-0.309458,-0.335673,-0.307234,-0.324500,-0.291576,-0.407253,-0.239280
Beta,0.982919,1.046639,1.063388,1.001575,1.052794,1.095672,1.000000
Information Ratio,0.299674,-0.006449,-0.284349,-0.360243,-0.233486,0.066730,NaN


**`Portfolio`** — a convenience wrapper that pre-fills `returns`, `benchmark`, `rf` and
`periods_per_year`; every function of chapters 2–8 is available as a method and gives exactly the
bare function's number.

In [47]:
pf = paris.Portfolio(rets, benchmark=spx, rf=tbill)
pf.sharpe(), pf.beta()

(FCNTX    0.951415
 AGTHX    0.829101
 FMAGX    0.744000
 AMCPX    0.786785
 DODGX    0.714116
 PRGFX    0.800508
 dtype: float64,
 FCNTX    0.982919
 AGTHX    1.046639
 FMAGX    1.063388
 AMCPX    1.001575
 DODGX    1.052794
 PRGFX    1.095672
 dtype: float64)

## 8. Tables

Every table cell is a call to a chapter 2–6 function: rows are metrics or periods, columns are funds,
benchmark last when supplied.

**`annualized_table`** — annualised return, volatility and Sharpe. As in the R reference package's table, the Sharpe numerator is the **geometric** annualised excess return (`geometric=True`), so it is a little lower than the arithmetic default of `sharpe` and `stats`.

In [48]:
paris.annualized_table(rets, spx, rf=tbill)

,FCNTX,AGTHX,FMAGX,AMCPX,DODGX,PRGFX,SPY
metric,,,,,,,
CAGR,0.155294,0.139927,0.126761,0.126653,0.124739,0.144709,0.140219
Volatility (ann.),0.149863,0.156751,0.159446,0.148492,0.164447,0.170421,0.143359
Sharpe,0.931765,0.793007,0.697654,0.748243,0.662701,0.756831,0.868233


**`calendar_table`** — the month grid of one fund with annual totals and the benchmark's annual return.

In [49]:
paris.calendar_table(fund, spx).tail()

,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,Annual,SPY
year,,,,,,,,,,,,,,
2021,-0.010740,0.015663,0.020656,0.070238,0.002225,0.041065,0.021322,0.045407,-0.059910,0.069570,-0.000993,0.013184,0.244363,0.287441
2022,-0.082090,-0.048894,0.032777,-0.115569,-0.012864,-0.087791,0.090977,-0.039972,-0.081838,0.046912,0.052278,-0.056975,-0.282613,-0.181717
2023,0.072787,-0.018420,0.058964,0.032355,0.025510,0.060412,0.041555,-0.009653,-0.031839,-0.009396,0.081978,0.039804,0.393324,0.261895
2024,0.048477,0.093853,0.028742,-0.046389,0.069099,0.044467,-0.014851,0.039196,0.020793,-0.003316,0.048954,-0.009425,0.359712,0.248851
2025,0.054208,-0.018044,-0.073148,0.007992,0.082755,0.067735,0.031719,0.002493,0.026109,0.010097,-0.000800,0.016607,0.217542,0.177151


**`capture_table`** — up/down capture, number and percentage ratios.

In [50]:
paris.capture_table(rets, spx)

,FCNTX,AGTHX,FMAGX,AMCPX,DODGX,PRGFX
metric,,,,,,
Up Capture,1.079566,1.117339,1.030588,0.880008,1.035983,1.252908
Down Capture,0.982469,1.013076,1.024007,1.007716,1.027263,1.017986
Up Number Ratio,0.924242,0.924242,0.886364,0.939394,0.886364,0.871212
Down Number Ratio,0.933333,0.966667,0.900000,0.933333,0.950000,0.916667
Up Percentage Ratio,0.500000,0.530303,0.522727,0.484848,0.568182,0.553030
Down Percentage Ratio,0.600000,0.466667,0.366667,0.483333,0.433333,0.483333


**`downside_table`** — the downside-risk statistics of chapters 3–4 in one table.

In [51]:
paris.downside_table(rets, spx, rf=tbill)

,FCNTX,AGTHX,FMAGX,AMCPX,DODGX,PRGFX,SPY
metric,,,,,,,
Semi Deviation,0.031917,0.033340,0.034311,0.032057,0.035083,0.036427,0.030924
Gain Deviation,0.026732,0.027927,0.027020,0.025771,0.029373,0.030101,0.025116
Loss Deviation,0.028516,0.030537,0.030981,0.029940,0.034030,0.034746,0.027667
Downside Deviation (MAR),0.029516,0.031497,0.032929,0.030782,0.033767,0.034316,0.029163
Downside Deviation (rf),0.026113,0.028101,0.029442,0.027442,0.030414,0.030950,0.025815
Downside Deviation (0),0.025593,0.027563,0.028915,0.026895,0.029879,0.030432,0.025277
Max Drawdown,-0.309458,-0.335673,-0.307234,-0.324500,-0.291576,-0.407253,-0.239280
VaR 95% (hist.),-0.063048,-0.075210,-0.070368,-0.068567,-0.071126,-0.084330,-0.062220
CVaR 95% (hist.),-0.085832,-0.093040,-0.094856,-0.090274,-0.098728,-0.101233,-0.083618


**`drawdown_summary`** — depth, length and recovery statistics across funds.

In [52]:
paris.drawdown_summary(rets, spx)

,FCNTX,AGTHX,FMAGX,AMCPX,DODGX,PRGFX,SPY
metric,,,,,,,
Max Drawdown,-0.309458,-0.335673,-0.307234,-0.324500,-0.291576,-0.407253,-0.239280
Average Drawdown,0.052589,0.053131,0.056329,0.057422,0.054767,0.063789,0.055792
Longest Drawdown (periods),25.000000,28.000000,26.000000,26.000000,18.000000,32.000000,24.000000
Average Length (periods),4.242424,4.176471,4.562500,4.724138,4.687500,4.933333,4.266667
Average Recovery (periods),2.333333,2.303030,2.677419,2.821429,2.718750,2.862069,2.466667
Current Drawdown,0.000000,-0.007860,-0.029047,-0.005858,0.000000,-0.025139,0.000000
Ulcer Index,0.075679,0.090013,0.085770,0.084247,0.061398,0.108503,0.056425
Pain Index,0.038269,0.046289,0.049006,0.043843,0.035139,0.054912,0.030148


**`rolling`** — apply any scalar function over a trailing window (here a 36-month Sharpe ratio);
`benchmark` and `rf` are passed through when the function takes them.

In [53]:
paris.rolling(rets, paris.sharpe, 36, rf=tbill).tail()

,FCNTX,AGTHX,FMAGX,AMCPX,DODGX,PRGFX
date,,,,,,
2025-08-31,1.368643,1.072616,0.966939,0.915619,0.663313,0.982433
2025-09-30,1.738730,1.400590,1.328898,1.238049,0.944997,1.353872
2025-10-31,1.658420,1.362586,1.204195,1.199429,0.730156,1.346709
2025-11-30,1.539601,1.244456,1.031492,1.096999,0.621603,1.238997
2025-12-31,1.815234,1.462735,1.222078,1.241004,0.816355,1.519640


## 9. Portfolio construction and return attribution

Weights are a one-time vector or a DataFrame of dated rows (a row dated *d* applies to returns
strictly after *d*); they must sum to 1. Between rebalances the portfolio drifts with buy-and-hold
value; `rebalance` resets to the target weights at the given calendar frequency.

In [54]:
w = [0.3, 0.2, 0.2, 0.1, 0.1, 0.1]           # target weights for the six funds
pd.Series(w, index=rets.columns, name="weight")

FCNTX    0.3
AGTHX    0.2
FMAGX    0.2
AMCPX    0.1
DODGX    0.1
PRGFX    0.1
Name: weight, dtype: float64

**`portfolio_return`** — the portfolio's return series from asset returns and weights, rebalanced quarterly here.

In [55]:
port = paris.portfolio_return(rets, w, rebalance="QE")
port.tail()

date
2025-08-31    0.008273
2025-09-30    0.023374
2025-10-31    0.013256
2025-11-30   -0.004962
2025-12-31    0.003413
Name: Portfolio, dtype: float64

**`contribution`** — each asset's per-period contribution (beginning-of-period weight × return); the rows sum to the portfolio return.

In [56]:
contrib = paris.contribution(rets, w, rebalance="QE")
contrib.tail()

,FCNTX,AGTHX,FMAGX,AMCPX,DODGX,PRGFX
date,,,,,,
2025-08-31,0.000754,0.003449,-0.001622,0.001039,0.003985,0.000669
2025-09-30,0.007855,0.005992,0.002351,0.002368,0.000218,0.004589
2025-10-31,0.003029,0.004987,-0.000497,0.002860,-0.000632,0.003508
2025-11-30,-0.000239,-0.001798,-0.003188,0.000632,0.001247,-0.001617
2025-12-31,0.004987,0.000209,-0.002066,-0.000601,0.001840,-0.000956


**`period_contributions`** — contributions linked over calendar spans (years here) so that they still add up to the span's portfolio return.

In [57]:
paris.period_contributions(contrib, "YE").tail()

,FCNTX,AGTHX,FMAGX,AMCPX,DODGX,PRGFX,Portfolio
date,,,,,,,
2021-12-31,0.073739,0.039334,0.055082,0.023606,0.030219,0.020208,0.242190
2022-12-31,-0.084708,-0.061817,-0.054662,-0.029238,-0.006387,-0.041897,-0.278710
2023-12-31,0.115681,0.074148,0.063139,0.031619,0.019192,0.043441,0.347220
2024-12-31,0.105731,0.057485,0.055635,0.021649,0.015163,0.029862,0.285526
2025-12-31,0.064298,0.040288,0.021314,0.018192,0.013566,0.016825,0.174484


**`brinson`** — Brinson–Fachler allocation, selection and interaction effects, one row per category and
a `Total` row, linked over the whole window (Cariño linking by default; `freq="YE"` gives one table per
year). Categories are matched by column name, so here the portfolio is two style sleeves — a growth fund
and a value fund — measured against the Russell 1000 Growth / Value indices at a 50/50 mix.

In [58]:
sleeves = rets[["FCNTX", "DODGX"]].set_axis(["Growth", "Value"], axis=1)   # portfolio: 60 / 40
style   = m[["IWF", "IWD"]].set_axis(["Growth", "Value"], axis=1)           # benchmark: 50 / 50
paris.brinson(sleeves, [0.6, 0.4], style, [0.5, 0.5], rebalance="QE", benchmark_rebalance="QE")

,Allocation,Selection,Interaction,Active
asset,,,,
Growth,0.308864,-0.500479,-0.094945,-0.286561
Value,0.320670,0.977340,-0.198359,1.099651
Total,0.629534,0.476861,-0.293304,0.813090


## 10. Risk budgeting

Euler decompositions of portfolio risk for **one** weight vector held constant: the contributions sum to
the volatility / VaR / CVaR of the fixed-weight portfolio (`portfolio_return(..., rebalance="ME")` on
monthly data); `pct=True` returns shares instead.

**`volatility_contribution`** — each asset's share of annualised portfolio volatility.

In [59]:
paris.volatility_contribution(rets, w, pct=True)

FCNTX    0.291167
AGTHX    0.204066
FMAGX    0.206066
AMCPX    0.096060
DODGX    0.093544
PRGFX    0.109097
Name: pct_contribution, dtype: float64

**`cvar_contribution`** — contribution to the portfolio's historical 95 % expected shortfall; the sum is the portfolio CVaR.

In [60]:
c = paris.cvar_contribution(rets, w)
c, c.sum(), paris.cvar(paris.portfolio_return(rets, w, rebalance="ME"))

(FCNTX   -0.025750
 AGTHX   -0.018608
 FMAGX   -0.018670
 AMCPX   -0.009027
 DODGX   -0.008308
 PRGFX   -0.009945
 Name: contribution, dtype: float64,
 -0.09030771235309999,
 -0.09030771235309999)

**`marginal_var`** — leave-one-out VaR: the VaR of the portfolio without each asset (remaining weights rescaled) minus the VaR of the full portfolio. Negative means the portfolio would be riskier without that asset.

In [61]:
paris.marginal_var(rets, w)

FCNTX   -0.001943
AGTHX    0.001446
FMAGX   -0.000681
AMCPX   -0.000283
DODGX   -0.001816
PRGFX    0.002063
Name: marginal, dtype: float64

## 11. Sample data

`paris.data.describe()` lists every column of the two frozen datasets with its full name, role and
window. The data is illustrative and plays no part in the library's oracle tests.

In [62]:
paris.data.describe()

,dataset,column,name,role,frequency,first,last
0,managers,FCNTX,Fidelity Contrafund,fund,monthly simple total returns,2010-01-31,2025-12-31
1,managers,AGTHX,American Funds Growth Fund of America,fund,monthly simple total returns,2010-01-31,2025-12-31
2,managers,FMAGX,Fidelity Magellan,fund,monthly simple total returns,2010-01-31,2025-12-31
3,managers,AMCPX,American Funds AMCAP,fund,monthly simple total returns,2010-01-31,2025-12-31
4,managers,DODGX,Dodge & Cox Stock,fund,monthly simple total returns,2010-01-31,2025-12-31
5,managers,PRGFX,T. Rowe Price Growth Stock,fund,monthly simple total returns,2010-01-31,2025-12-31
6,managers,SPY,S&P 500 (ETF total-return proxy),benchmark,monthly simple total returns,2010-01-31,2025-12-31
7,managers,IWF,Russell 1000 Growth (ETF total-return proxy),benchmark,monthly simple total returns,2010-01-31,2025-12-31
8,managers,IWD,Russell 1000 Value (ETF total-return proxy),benchmark,monthly simple total returns,2010-01-31,2025-12-31
9,managers,TBILL3M,"3-month Treasury bill yield, per month",risk-free,monthly simple total returns,2010-01-31,2025-12-31
